In [9]:
# NOTE:
# This notebook is ONE-TIME setup.
# Do not edit unless changing data sources or seasons.

In [1]:
# ------------------------------------------------------------
# Make paths work no matter where the notebook is run from
# (sets project root as the working directory)
# ------------------------------------------------------------

from pathlib import Path
import os

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(PROJECT_ROOT)

print("Working directory set to:", Path.cwd())

Working directory set to: /Users/darellsam/Desktop/SuperBowl LX Predictions


In [2]:
# ------------------------------------------------------------
# Setup imports + folder constants (run once per kernel)
# ------------------------------------------------------------

from pathlib import Path
import os
import pandas as pd
import nfl_data_py as nfl

RAW_DIR = Path("data/raw")
NFLVERSE_DIR = RAW_DIR / "nflverse"
ELO_DIR = RAW_DIR / "elo"
ODDS_DIR = RAW_DIR / "odds"

NFLVERSE_DIR.mkdir(parents=True, exist_ok=True)
ELO_DIR.mkdir(parents=True, exist_ok=True)
ODDS_DIR.mkdir(parents=True, exist_ok=True)

print("Folders ready:")
print(" -", NFLVERSE_DIR)
print(" -", ELO_DIR)
print(" -", ODDS_DIR)

print("nfl_data_py import OK:", nfl.__version__ if hasattr(nfl, "__version__") else "version unknown")

Folders ready:
 - data/raw/nflverse
 - data/raw/elo
 - data/raw/odds
nfl_data_py import OK: version unknown


In [3]:
# ------------------------------------------------------------
# Dataset 1: nflverse Play-by-Play (PBP) -> Parquet cache
# ------------------------------------------------------------

PBP_PATH = NFLVERSE_DIR / "pbp_1999_2025.parquet"

if PBP_PATH.exists():
    print("PBP already cached:", PBP_PATH)
else:
    seasons = list(range(1999, 2026))  # 1999-2025 seasons
    pbp = nfl.import_pbp_data(seasons)
    pbp.to_parquet(PBP_PATH, engine="pyarrow")
    print("PBP cached:", PBP_PATH, "| shape:", pbp.shape)

PBP already cached: data/raw/nflverse/pbp_1999_2025.parquet


In [4]:
# ------------------------------------------------------------
# Dataset 2: nflverse Schedules (game-level results + metadata)
# ------------------------------------------------------------

SCHED_PATH = NFLVERSE_DIR / "schedules_1999_2025.parquet"

if SCHED_PATH.exists():
    print("Schedules already cached:", SCHED_PATH)
else:
    years = list(range(1999, 2026))  # 1999-2025 seasons
    schedules = nfl.import_schedules(years)
    schedules.to_parquet(SCHED_PATH, engine="pyarrow")
    print("Schedules cached:", SCHED_PATH, "| shape:", schedules.shape)

Schedules already cached: data/raw/nflverse/schedules_1999_2025.parquet


In [5]:
# ------------------------------------------------------------
# Dataset 3: FiveThirtyEight Elo (Kaggle mirror) CSV -> Parquet
# ------------------------------------------------------------

RAW_ELO_CSV = ELO_DIR / "nfl_games.csv"
ELO_PARQUET = ELO_DIR / "nfl_games.parquet"

if ELO_PARQUET.exists():
    print("Elo already cached:", ELO_PARQUET)
else:
    if not RAW_ELO_CSV.exists():
        raise FileNotFoundError(
            f"Missing {RAW_ELO_CSV}. Put Kaggle 'nfl_games.csv' into {ELO_DIR} first."
        )
    elo_df = pd.read_csv(RAW_ELO_CSV)
    elo_df.to_parquet(ELO_PARQUET, engine="pyarrow")
    print("Elo cached:", ELO_PARQUET, "| shape:", elo_df.shape)

Elo already cached: data/raw/elo/nfl_games.parquet


In [6]:
# ------------------------------------------------------------
# Dataset 4: Betting odds (SpreadSpoke) CSV -> Parquet
# ------------------------------------------------------------

RAW_ODDS_CSV = ODDS_DIR / "spreadspoke_scores.csv"
ODDS_PARQUET = ODDS_DIR / "odds.parquet"

if ODDS_PARQUET.exists():
    print("Odds already cached:", ODDS_PARQUET)
else:
    if not RAW_ODDS_CSV.exists():
        raise FileNotFoundError(
            f"Can't find: {RAW_ODDS_CSV}\n"
            f"Make sure you placed 'spreadspoke_scores.csv' inside: {ODDS_DIR.resolve()}"
        )
    odds_df = pd.read_csv(RAW_ODDS_CSV)
    odds_df.to_parquet(ODDS_PARQUET, engine="pyarrow")
    print("Odds cached:", ODDS_PARQUET, "| shape:", odds_df.shape)

Odds already cached: data/raw/odds/odds.parquet


In [7]:
# ------------------------------------------------------------
# Dataset 5: nflverse Rosters & Availability (ONE-TIME)
#
# Notes:
# - Season rosters: usually available for all years requested.
# - Weekly rosters and depth charts: not available for all years upstream.
#   This cell downloads year-by-year and skips missing years.
# - Also cleans mixed-type numeric columns to avoid pyarrow failures.
# ------------------------------------------------------------

ROSTER_DIR = NFLVERSE_DIR

SEASON_ROSTERS_PATH = ROSTER_DIR / "season_rosters_1999_2025.parquet"
WEEKLY_ROSTERS_PATH = ROSTER_DIR / "weekly_rosters_1999_2025.parquet"
DEPTH_CHARTS_PATH   = ROSTER_DIR / "depth_charts_2001_2025.parquet"

years = list(range(1999, 2026))  # 1999-2025 seasons

def _to_nullable_int(df, col):
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int64")
    return df

def clean_roster_df(df):
    for col in [
        "jersey_number",
        "draft_number",
        "draft_round",
        "draft_year",
        "birth_year",
        "height",
        "weight",
    ]:
        df = _to_nullable_int(df, col)
    return df

# -------------------------
# Season rosters
# -------------------------
if SEASON_ROSTERS_PATH.exists():
    print("Season rosters already cached:", SEASON_ROSTERS_PATH)
else:
    print("Downloading season rosters...")
    season_rosters = nfl.import_seasonal_rosters(years)
    season_rosters = clean_roster_df(season_rosters)
    season_rosters.to_parquet(SEASON_ROSTERS_PATH, engine="pyarrow")
    print("Season rosters cached:", season_rosters.shape, "|", SEASON_ROSTERS_PATH)

# -------------------------
# Weekly rosters
# -------------------------
if WEEKLY_ROSTERS_PATH.exists():
    print("Weekly rosters already cached:", WEEKLY_ROSTERS_PATH)
else:
    print("Downloading weekly rosters (skipping years not available)...")
    weekly_parts = []
    missing_weekly_years = []

    for y in years:
        try:
            tmp = nfl.import_weekly_rosters([y])
            tmp = clean_roster_df(tmp)
            weekly_parts.append(tmp)
            print(f"  weekly rosters: {y} OK")
        except Exception as e:
            missing_weekly_years.append(y)
            print(f"  weekly rosters: {y} SKIP ({type(e).__name__})")

    if not weekly_parts:
        raise RuntimeError("Could not download weekly rosters for any year (upstream unavailable).")

    weekly_rosters = pd.concat(weekly_parts, ignore_index=True)
    weekly_rosters.to_parquet(WEEKLY_ROSTERS_PATH, engine="pyarrow")

    print("Weekly rosters cached:", weekly_rosters.shape, "|", WEEKLY_ROSTERS_PATH)
    if missing_weekly_years:
        print(f"Skipped {len(missing_weekly_years)} weekly roster years. First few:", missing_weekly_years[:10])

# -------------------------
# Depth charts
# -------------------------
if DEPTH_CHARTS_PATH.exists():
    print("Depth charts already cached:", DEPTH_CHARTS_PATH)
else:
    print("Downloading depth charts (skipping years not available)...")
    depth_parts = []
    missing_depth_years = []

    for y in years:
        try:
            tmp = nfl.import_depth_charts([y])
            depth_parts.append(tmp)
            print(f"  depth charts: {y} OK")
        except Exception as e:
            missing_depth_years.append(y)
            print(f"  depth charts: {y} SKIP ({type(e).__name__})")

    if not depth_parts:
        raise RuntimeError("Could not download depth charts for any year (upstream unavailable).")

    depth_charts = pd.concat(depth_parts, ignore_index=True)
    depth_charts.to_parquet(DEPTH_CHARTS_PATH, engine="pyarrow")

    print("Depth charts cached:", depth_charts.shape, "|", DEPTH_CHARTS_PATH)
    if missing_depth_years:
        print(f"Skipped {len(missing_depth_years)} depth chart years. First few:", missing_depth_years[:10])

Season rosters already cached: data/raw/nflverse/season_rosters_1999_2025.parquet
Weekly rosters already cached: data/raw/nflverse/weekly_rosters_1999_2025.parquet
Depth charts already cached: data/raw/nflverse/depth_charts_2001_2025.parquet


In [8]:
# ------------------------------------------------------------
# Verify cached files exist + show sizes
# ------------------------------------------------------------

paths = [
    NFLVERSE_DIR / "pbp_1999_2025.parquet",
    NFLVERSE_DIR / "schedules_1999_2025.parquet",
    ELO_DIR / "nfl_games.parquet",
    ODDS_DIR / "odds.parquet",
    NFLVERSE_DIR / "season_rosters_1999_2025.parquet",
    NFLVERSE_DIR / "weekly_rosters_1999_2025.parquet",
    NFLVERSE_DIR / "depth_charts_2001_2025.parquet",
]

for p in paths:
    exists = p.exists()
    size_mb = (p.stat().st_size / (1024**2)) if exists else 0
    print(f"{'OK' if exists else 'MISSING'} | {p} | {size_mb:,.1f} MB")

OK | data/raw/nflverse/pbp_1999_2025.parquet | 378.3 MB
OK | data/raw/nflverse/schedules_1999_2025.parquet | 0.5 MB
OK | data/raw/elo/nfl_games.parquet | 0.4 MB
OK | data/raw/odds/odds.parquet | 0.2 MB
OK | data/raw/nflverse/season_rosters_1999_2025.parquet | 4.3 MB
OK | data/raw/nflverse/weekly_rosters_1999_2025.parquet | 9.7 MB
OK | data/raw/nflverse/depth_charts_2001_2025.parquet | 11.4 MB
